# 04 — S3 State Storage

Validate S3 read/write for production state persistence:
- Connect to Scaleway S3 (`my-imagestore` bucket)
- Write state under `growth-agent/` prefix
- Read back and verify round-trip
- List existing keys

In [1]:
import sys
sys.path.insert(0, '..')

from dotenv import load_dotenv
import os

load_dotenv('../.env')

SCW_ACCESS_KEY = os.getenv('SCW_ACCESS_KEY')
SCW_SECRET_KEY = os.getenv('SCW_SECRET_KEY')
S3_BUCKET = os.getenv('S3_BUCKET', 'my-imagestore')

print(f'SCW credentials: {"yes" if SCW_ACCESS_KEY and SCW_SECRET_KEY else "NO — add to .env"}')
print(f'Bucket: {S3_BUCKET}')

SCW credentials: yes
Bucket: my-imagestore


## 1. Connect to S3

In [2]:
from agent.storage import S3Storage

s3 = S3Storage(
    bucket=S3_BUCKET,
    prefix='growth-agent/',
    access_key=SCW_ACCESS_KEY,
    secret_key=SCW_SECRET_KEY,
)

# List existing keys
keys = s3.list_keys()
print(f'Existing keys under growth-agent/: {keys if keys else "(empty — first run)"}')

Existing keys under growth-agent/: (empty — first run)


## 2. Write Test Data

In [3]:
from agent.models import Strategy

# Write default strategy to S3
strategy = Strategy()
s3.write('strategy.json', strategy)
print(f'Wrote strategy.json to S3')
print(f'Content pillars: {strategy.content_pillars}')

Wrote strategy.json to S3
Content pillars: ['Politische Ökonomie & Spieltheorie', 'Blockchain & Web3 (NFTs, x402, Smart Contracts)', 'Quantencomputing & QML', 'AI-Tools & Infrastruktur']


## 3. Read Back and Verify

In [4]:
loaded = s3.read('strategy.json')
print(f'Read back strategy.json from S3:')
print(f'  content_pillars: {loaded["content_pillars"]}')
print(f'  channels: {loaded["channels"]}')
print(f'  tone: {loaded["tone"]}')

# Verify it parses back into a Strategy model
loaded_strategy = Strategy(**loaded)
assert loaded_strategy.content_pillars == strategy.content_pillars
print('\n✅ Round-trip verified — Pydantic model intact.')

Read back strategy.json from S3:
  content_pillars: ['Politische Ökonomie & Spieltheorie', 'Blockchain & Web3 (NFTs, x402, Smart Contracts)', 'Quantencomputing & QML', 'AI-Tools & Infrastruktur']
  channels: ['mastodon', 'bluesky']
  tone: insightful, technical, opinionated, accessible

✅ Round-trip verified — Pydantic model intact.


## 4. Upload Local State to S3

Copy insights and content queue from local state (notebooks 01-03) to S3.

In [5]:
from agent.storage import LocalStorage

local = LocalStorage(base_dir='../state')

# Upload all local state files to S3
for key in local.list_keys():
    data = local.read(key)
    if data is not None:
        s3.write(key, data)
        print(f'  Uploaded {key}')

# Verify
s3_keys = s3.list_keys()
print(f'\nAll S3 keys: {s3_keys}')

  Uploaded insights.json
  Uploaded content_queue.json
  Uploaded llm_analysis.json

All S3 keys: ['content_queue.json', 'insights.json', 'llm_analysis.json', 'strategy.json']


## 5. Read Non-Existent Key

In [6]:
missing = s3.read('does-not-exist.json')
print(f'Reading missing key: {missing}')
assert missing is None
print('✅ Returns None for missing keys.')

Reading missing key: None
✅ Returns None for missing keys.


In [7]:
print('Done — S3 state storage validated.')

Done — S3 state storage validated.
